# 03. 사전학습 영상 분류 모델 살펴보기

Food101에서는 ImageNet으로 사전학습된 2D ResNet을 가져다 썼다. 영상도 똑같은 전이학습 아이디어가
통하는데, 대신 **Kinetics-400** (40만 개 유튜브 클립, 400개 동작 클래스)으로 사전학습된
3D 모델을 가져다 쓴다.

`torchvision.models.video`에 바로 쓸 수 있는 대표적인 모델 3가지가 있다:

| 모델 | 한 줄 설명 | 특징 |
|---|---|---|
| `r3d_18` | 2D ResNet-18을 그대로 3D convolution으로 확장 | 구조가 가장 단순, 이해하기 쉬움 |
| `mc3_18` | 앞쪽 레이어는 3D, 뒤쪽은 2D로 섞은 "Mixed Convolution" | 3D conv가 적어서 더 가볍고 빠름 |
| `r2plus1d_18` | 3D conv를 (공간 2D + 시간 1D)로 쪼갠 "(2+1)D" | 파라미터는 비슷한데 정확도가 보통 제일 좋음 |

여기서는 구조가 2D ResNet과 가장 비슷해서 비교하기 쉬운 `r3d_18`을 기본으로 쓰고,
나머지 둘도 가볍게 비교해본다.

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
from torchvision.models.video import (
    r3d_18, R3D_18_Weights,
    mc3_18, MC3_18_Weights,
    r2plus1d_18, R2Plus1D_18_Weights,
)


def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


models_info = {
    "r3d_18": (r3d_18, R3D_18_Weights.KINETICS400_V1),
    "mc3_18": (mc3_18, MC3_18_Weights.KINETICS400_V1),
    "r2plus1d_18": (r2plus1d_18, R2Plus1D_18_Weights.KINETICS400_V1),
}

for name, (builder, weights) in models_info.items():
    m = builder(weights=weights)
    print(f"{name:<12} 파라미터 {count_params(m):5.1f}M | Kinetics-400 top-1 acc "
          f"{weights.meta['_metrics']['Kinetics-400']['acc@1']:.1f}")


Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /home/noni/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:07<00:00, 18.3MB/s] 


r3d_18       파라미터  33.4M | Kinetics-400 top-1 acc 63.2
Downloading: "https://download.pytorch.org/models/mc3_18-a90a0ba3.pth" to /home/noni/.cache/torch/hub/checkpoints/mc3_18-a90a0ba3.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 60.6MB/s]


mc3_18       파라미터  11.7M | Kinetics-400 top-1 acc 64.0
Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to /home/noni/.cache/torch/hub/checkpoints/r2plus1d_18-91a641e6.pth


100%|██████████| 120M/120M [00:02<00:00, 61.3MB/s] 


r2plus1d_18  파라미터  31.5M | Kinetics-400 top-1 acc 67.5


## 3-1. 입력 전처리 확인

각 가중치 객체(`Weights`)에는 모델이 기대하는 입력 크기/정규화 값이 들어있다.
`video_utils.py`에서 쓰는 `KINETICS_MEAN/STD`, 112x112, 16프레임 설정이 실제로 일치하는지
여기서 확인한다.

In [2]:
tf = R3D_18_Weights.KINETICS400_V1.transforms()
print(tf)


VideoClassification(
    crop_size=[112, 112]
    resize_size=[128, 171]
    mean=[0.43216, 0.394666, 0.37645]
    std=[0.22803, 0.22145, 0.216989]
    interpolation=InterpolationMode.BILINEAR
)


공식 전처리는 128x171로 리사이즈한 뒤 112x112로 center-crop하지만, `video_utils.py`에서는
구현을 단순하게 하려고 **곧바로 112x112로 리사이즈**한다 (가로세로 비율이 살짝 눌린다).
평균/표준편차 값은 위 출력과 동일하게 맞춰뒀다. 정확도를 더 끌어올리고 싶다면 공식 전처리
방식(resize → center crop)으로 바꿔보는 것도 좋은 연습이 된다.

## 3-2. 분류 head 교체하기

Food101 때 `model.fc = nn.Linear(in_features, 101)`로 바꿨던 것과 완전히 동일한 패턴이다.
Kinetics-400용 400-way head를 떼어내고, 우리 데이터셋의 클래스 수로 바꾼다.

In [3]:
DATA_DIR = Path("data/UCF101_subset")
CLASSES = sorted(p.name for p in (DATA_DIR / "train").iterdir() if p.is_dir())
NUM_CLASSES = len(CLASSES)  # 01 노트북에서 받은 UCF101 서브셋 클래스 수
print(f"NUM_CLASSES = {NUM_CLASSES}")

model = r3d_18(weights=R3D_18_Weights.KINETICS400_V1)
print("교체 전 head:", model.fc)

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
print("교체 후 head:", model.fc)


NUM_CLASSES = 10
교체 전 head: Linear(in_features=512, out_features=400, bias=True)
교체 후 head: Linear(in_features=512, out_features=10, bias=True)


## 3-3. 어디까지 freeze 할지 정하기

데이터가 아주 적을 때(클래스당 12개 학습 영상)는 백본 전체를 학습시키면 바로 과적합된다.
Food101 때처럼 **뒤쪽 블록(layer4)과 새 fc만 학습**하고, 나머지는 얼려서 Kinetics-400에서
배운 저수준 시공간 특징을 그대로 재사용한다.

In [4]:
for name, param in model.named_parameters():
    param.requires_grad = name.startswith("layer4") or name.startswith("fc")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
total = count_params(model)
print(f"학습 대상 파라미터: {trainable:.1f}M / 전체 {total:.1f}M")


학습 대상 파라미터: 24.9M / 전체 33.2M


다음 노트북(`04_finetune_video_classifier.ipynb`)에서 이 모델을 실제로 학습시킨다.